# Transport Analytics Notebook

This notebook is the supporting walkthrough for the final report. It follows the PostgreSQL-backed project flow, reads the generated result tables under `report/results/`, and stays intentionally narrower than the paper so it does not become a competing narrative.


## Objective

The project compares Paris and NYC transport demand with four final methods:

- temporal profiling
- lag-based forecasting
- anomaly detection
- contributor and city-structure comparison


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS = ROOT / 'report' / 'results'
FIGURES = ROOT / 'report' / 'figures'
import sys
sys.path.insert(0, str(ROOT / 'src'))
plt.style.use('seaborn-v0_8-whitegrid')
RESULTS


## Data and architecture

The project keeps uploaded transport data in PostgreSQL and exposes reporting-ready views through the `transport` schema. The official execution path is PostgreSQL-first: `scripts/run_stage_workflow.py` reads those views, generates the analytical outputs, and writes the derived CSV files consumed here.

The notebook does not rerun the full workflow by default. Its role is to inspect the generated outputs, mirror the report structure at a high level, and keep the interpretation consistent with the paper.


In [ ]:
from transport_analytics import PostgresConfig

required = ['PGHOST', 'PGPORT', 'PGDATABASE', 'PGUSER', 'PGPASSWORD']
if all(os.getenv(key) for key in required):
    pg = PostgresConfig.from_env()
    print(f'PostgreSQL target: {pg.user}@{pg.host}:{pg.port}/{pg.database} [{pg.schema}]')
else:
    print('PostgreSQL environment not loaded in this session. Using generated outputs from report/results/.')


In [ ]:
summary = json.loads((RESULTS / 'analysis_summary.json').read_text())
forecast_overall = pd.read_csv(RESULTS / 'forecast_metrics_overall.csv')
forecast_by_city = pd.read_csv(RESULTS / 'forecast_metrics_by_city.csv')
anomaly_rates = pd.read_csv(RESULTS / 'anomaly_rates.csv')
city_structure = pd.read_csv(RESULTS / 'city_structure_summary.csv')
top_contributors = pd.read_csv(RESULTS / 'top_contributors.csv')
yearly_totals = pd.read_csv(RESULTS / 'temporal_yearly_totals.csv')
monthly_profile = pd.read_csv(RESULTS / 'temporal_monthly_profile.csv')
forecast_predictions = pd.read_csv(RESULTS / 'forecast_predictions.csv', parse_dates=['date'])
anomaly_flags = pd.read_csv(RESULTS / 'anomaly_flags.csv', parse_dates=['date'])
summary


## Methods and reported outputs

The paper focuses on four methods and this notebook follows the same order:

- temporal profiling from cleaned daily demand aggregates
- lag-based forecasting from the reporting-ready daily series
- anomaly detection from the same daily baseline
- contributor and city-structure analysis from cleaned station and regional summaries

## Results

### Temporal patterns


In [ ]:
yearly_totals


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for city, group in monthly_profile.groupby('city'):
    avg = group.groupby('month', as_index=False)['avg_value'].mean()
    ax.plot(avg['month'], avg['avg_value'], marker='o', linewidth=2, label=city)
ax.set_title('Average monthly demand by city')
ax.set_xlabel('Month')
ax.set_ylabel('Average daily demand')
ax.legend()
plt.show()


### Forecasting


In [ ]:
forecast_overall, forecast_by_city


In [ ]:
plot_df = forecast_predictions.groupby(['date', 'city'], as_index=False)[['value', 'prediction']].sum().sort_values('date')
fig, ax = plt.subplots(figsize=(9, 4))
for city, group in plot_df.groupby('city'):
    ax.plot(group['date'], group['value'], linewidth=2, label=f'{city} actual')
    ax.plot(group['date'], group['prediction'], linestyle='--', label=f'{city} predicted')
ax.set_title('Forecast: actual vs predicted demand')
ax.set_xlabel('Date')
ax.set_ylabel('Demand')
ax.legend(ncol=2, fontsize=8)
plt.show()


### Anomaly detection


In [ ]:
anomaly_rates


In [ ]:
anomaly_flags[['date', 'city', 'region', 'value', 'zscore_30']].head(15)


### Contributors and city structure


In [ ]:
city_structure


In [ ]:
top_contributors.head(12)


## Discussion and limitations

- Paris carries the higher average daily volume in the cleaned reporting scope, while NYC shows a stronger relative weekday-weekend contrast.
- The current lag-based forecasting baseline performs better on NYC than on Paris, which is consistent with the lower NYC error metrics in `forecast_metrics_by_city.csv`.
- Paris also shows the higher anomaly rate in the generated outputs, so the anomaly tables should be read as a screening layer for disruption periods rather than a causal explanation.
- This notebook stays deliberately concise: the SQL-backed workflow and the report remain the primary source of truth for methods, evidence, and final claims.
